In [1]:
# importing required libraries
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# loading the environment variables
load_dotenv(override=True) # load the environment variables from the .env file
openai_api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
claude_api_key = os.getenv('CLAUDE_API_KEY')


# validate whether the API keys are loaded correctly
# OpenAI
if not openai_api_key:
    print("OPENAI_API_KEY is missing")
elif not openai_api_key.startswith("sk-proj-"):
    print("OPENAI_API_KEY is set, but does not start with sk-proj-")
else:
    print(f"OPENAI_API_KEY loaded, starts with {openai_api_key[:8]}")

# Gemini (this course also uses GOOGLE_API_KEY)
if not gemini_api_key:
    print("GEMINI_API_KEY is missing — check the name in .env")
elif not gemini_api_key.startswith(("AIz", "AQ.")):
    print("GEMINI_API_KEY is set, but does not start with AIz or AQ.")
else:
    print(f"GEMINI_API_KEY loaded, starts with {gemini_api_key[:4]}")

# Claude
if not claude_api_key:
    print("CLAUDE_API_KEY is missing — check the name in .env")
elif not claude_api_key.startswith("sk-ant-"):
    print("CLAUDE_API_KEY is set, but does not start with sk-ant-")
else:
    print(f"CLAUDE_API_KEY loaded, starts with {claude_api_key[:8]}")

OPENAI_API_KEY loaded, starts with sk-proj-
GEMINI_API_KEY loaded, starts with AIza
CLAUDE_API_KEY loaded, starts with sk-ant-a


In [3]:
class PromptBuilder:
    """Fetched page: url + text. Builds chat messages for summarization."""

    SYSTEM_PROMPT = "summarize the content of a website"
    USER_PROMPT_PREFIX = "provide me highlights of the news in bullet points from the content of website:"
    
    def __init__(self, url: str):
        self.url = url
        self.text = fetch_website_contents(url)
        
    def messages(self) -> list[dict]:
        return [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": self.USER_PROMPT_PREFIX + self.text},
        ]

In [4]:
class ChatModel:
    """Any OpenAI-compatible endpoint (Gemini, Ollama, OpenAI)."""

    def __init__(self, name:str, client: OpenAI, model: str):
        self.name = name
        self.client = client
        self.model = model
        
    def complete(self, messages: list[dict]) -> str:
        # send the messages to the model and return the response
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        return response.choices[0].message.content

In [5]:
class WebsiteSummarizer:
    """Orchestrates: fetch URL → messages → model → markdown."""

    def __init__(self, chat_model: ChatModel):
        self.chat_model = chat_model

    def summarize(self, url: str) -> str:
        promptBuilder = PromptBuilder(url)
        payload = promptBuilder.messages() # build the messages for the model
        return self.chat_model.complete(payload) # return the model's response

    def display(self, url: str) -> None:
        print(f"{self.chat_model.name} ({self.chat_model.model})\n{url}\n")
        display(Markdown(self.summarize(url))) # display the model's response in markdown format

## **Gemini Chat Completions API**

In [6]:
gemini_model = ChatModel(
    name = 'Gemini',
    client = OpenAI(
        base_url = "https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key = gemini_api_key,
    ),
    model="gemini-3.1-flash-lite",
)

gemini_summarizer = WebsiteSummarizer(gemini_model)
gemini_summarizer.display("https://www.cnn.com/2026/08/30/world/live-news/nepal-china-flood")


Gemini (gemini-3.1-flash-lite)
https://www.cnn.com/2026/08/30/world/live-news/nepal-china-flood



Based on the title and context provided for the CNN report, here are the key highlights regarding the flooding in China and Nepal:

*   **Rising Death Toll:** The number of confirmed fatalities from the flooding in China and Nepal has continued to increase as the situation develops.
*   **Ongoing Search and Rescue:** Emergency teams and rescuers are actively searching for missing individuals, specifically focusing on workers who were caught in the path of the floods.
*   **Regional Impact:** The disaster is affecting both China and Nepal, with significant damage and hazardous conditions reported across the impacted areas.
*   **Critical Conditions:** Heavy rainfall and subsequent flooding have triggered search operations in difficult terrain, complicating efforts to reach those still missing.

*(Note: The text provided in your prompt was primarily the navigation and ad-feedback interface of the CNN website; the summary above is based on the subject matter defined by the page title.)*

## **OLLAMA Chat Completions API**

In [7]:
# test ollama server is up and running
import requests

OLLAMA_URL = "http://localhost:11434/"
print(requests.get(OLLAMA_URL).content)

b'Ollama is running'


https://ollama.com/library/ for exploring different models available in ollama portal

In [8]:
# pull the model
import ollama
ollama.pull("llama3.2:1b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [9]:
!ollama list

NAME           ID              SIZE      MODIFIED               
llama3.2:1b    baf6a787fdff    1.3 GB    Less than a second ago    


In [41]:
llama_model = ChatModel(
    name="Ollama",
    client=OpenAI(base_url = 'http://localhost:11434/' + "v1"),
    model="llama3.2:1b",
)

llama_summarizer = WebsiteSummarizer(llama_model)
llama_summarizer.display("https://www.bbc.com/news")

Ollama (llama3.2:1b)
https://www.bbc.com/news



Here are the highlights of the news in bullet points:

**US and International**

* US strikes Iranian launchers on Larak Island, killing some civilians
* Iran claims the attack, marking the first known US strike on Iran since late July
* US opens fire on Iranian launchers on Larak Island in response to suspected attacks on US Navy ships

**News and Politics**

* US and its allies are investigating possible cyber attacks on US and Iranian targets
* US and NATO officials condemn Iran's actions, marking a escalation of tensions in the region
* UK, Japan, and other Western countries join US warning calls to prevent Iranian missiles from being transported to the region to be tested near ships in the Strait of Hormuz

**Business and Technology**

* News of the US strike on Iranian launchers raises concerns about the impact on global trade and energy supply
* India and China deploy experts to assist with the rescue of those trapped in the hydropower tunnels (Nepal)
* Chinese telecom giant Huawei under fire over alleged involvement with Iranian intelligence

Fun fact demo

In [ ]:
display(Markdown(llama_model.complete([{"role": "user", "content": "Tell me a fun fact"}])))

Here's a fun fact: There is a type of jellyfish that is immortal. The Turritopsis dohrnii, also known as the "immortal jellyfish," is a species of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated infinitely, making the Turritopsis dohrnii theoretically immortal.